In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

# Load data
url = "https://raw.githubusercontent.com/waico/SKAB/master/data/valve1/1.csv"
df = pd.read_csv(url, sep=';', index_col='datetime', parse_dates=True)

# Original features
feature_columns = ['Accelerometer1RMS', 'Accelerometer2RMS', 
                   'Current', 'Pressure', 'Temperature', 
                   'Thermocouple', 'Voltage', 'Volume Flow RateRMS']

X_raw = df[feature_columns]
y_true = df['anomaly']

print("Original data shape:", X_raw.shape)
print("We start with", X_raw.shape[1], "features")
print("Goal: engineer better features to improve anomaly detection")

Original data shape: (1145, 8)
We start with 8 features
Goal: engineer better features to improve anomaly detection


In [2]:
def engineer_features(df, window=10):
    """
    Takes raw sensor data and creates richer features.
    For each sensor we add:
    - Rolling mean: average over last N seconds
    - Rolling std: how much it varied over last N seconds  
    - Rate of change: how much it changed from last second
    """
    df_features = df[feature_columns].copy()
    
    # Focus extra attention on strongest sensors
    priority_sensors = ['Pressure', 'Volume Flow RateRMS', 
                       'Current', 'Temperature']
    
    for sensor in feature_columns:
        # Rolling mean — what's the trend?
        df_features[f'{sensor}_mean_{window}s'] = (
            df[sensor].rolling(window=window).mean()
        )
        
        # Rolling std — how stable is it?
        df_features[f'{sensor}_std_{window}s'] = (
            df[sensor].rolling(window=window).std()
        )
        
        # Rate of change — how fast is it moving?
        df_features[f'{sensor}_roc'] = df[sensor].diff()
    
    # Drop rows with NaN values created by rolling window
    df_features = df_features.dropna()
    
    print(f"Original features: 8")
    print(f"Engineered features: {df_features.shape[1]}")
    print(f"Rows after dropping NaN: {df_features.shape[0]}")
    print(f"\nNew feature names:")
    for col in df_features.columns:
        print(f"  {col}")
    
    return df_features

# Engineer features with window=10
X_engineered = engineer_features(df, window=10)

Original features: 8
Engineered features: 32
Rows after dropping NaN: 1136

New feature names:
  Accelerometer1RMS
  Accelerometer2RMS
  Current
  Pressure
  Temperature
  Thermocouple
  Voltage
  Volume Flow RateRMS
  Accelerometer1RMS_mean_10s
  Accelerometer1RMS_std_10s
  Accelerometer1RMS_roc
  Accelerometer2RMS_mean_10s
  Accelerometer2RMS_std_10s
  Accelerometer2RMS_roc
  Current_mean_10s
  Current_std_10s
  Current_roc
  Pressure_mean_10s
  Pressure_std_10s
  Pressure_roc
  Temperature_mean_10s
  Temperature_std_10s
  Temperature_roc
  Thermocouple_mean_10s
  Thermocouple_std_10s
  Thermocouple_roc
  Voltage_mean_10s
  Voltage_std_10s
  Voltage_roc
  Volume Flow RateRMS_mean_10s
  Volume Flow RateRMS_std_10s
  Volume Flow RateRMS_roc


In [3]:
# Train Isolation Forest on engineered features
model_v2 = IsolationForest(
    contamination=0.35,
    random_state=42
)

# Align y_true with engineered features
# (we dropped first 10 rows due to rolling window)
y_true_aligned = y_true[X_engineered.index]

# Train on engineered features
model_v2.fit(X_engineered)

# Predict
predictions_v2 = model_v2.predict(X_engineered)
y_pred_v2 = (predictions_v2 == -1).astype(int)

print("MODEL V2 PERFORMANCE (with engineered features)")
print("="*50)
print(classification_report(y_true_aligned, y_pred_v2,
                            target_names=['Normal', 'Anomaly']))

cm_v2 = confusion_matrix(y_true_aligned, y_pred_v2)
print("Confusion Matrix:")
print(f"True Negatives  (correctly said normal):    {cm_v2[0][0]}")
print(f"False Positives (wrongly said anomaly):     {cm_v2[1][0]}")
print(f"False Negatives (missed real anomaly):      {cm_v2[0][1]}")
print(f"True Positives  (correctly caught anomaly): {cm_v2[1][1]}")

print("\n--- COMPARISON ---")
print(f"V1 (raw features):         Precision=0.43, Recall=0.43, F1=0.43")

MODEL V2 PERFORMANCE (with engineered features)
              precision    recall  f1-score   support

      Normal       0.68      0.69      0.68       734
     Anomaly       0.42      0.42      0.42       402

    accuracy                           0.59      1136
   macro avg       0.55      0.55      0.55      1136
weighted avg       0.59      0.59      0.59      1136

Confusion Matrix:
True Negatives  (correctly said normal):    503
False Positives (wrongly said anomaly):     235
False Negatives (missed real anomaly):      231
True Positives  (correctly caught anomaly): 167

--- COMPARISON ---
V1 (raw features):         Precision=0.43, Recall=0.43, F1=0.43


In [6]:
# Fix 1: Use only strong sensors
strong_sensors = ['Pressure', 'Volume Flow RateRMS', 
                  'Current', 'Temperature']

def engineer_features_v2(df, sensors, window=10):
    df_features = pd.DataFrame(index=df.index)
    
    for sensor in sensors:
        # Raw value
        df_features[sensor] = df[sensor]
        
        # Rolling mean — trend over last N seconds
        df_features[f'{sensor}_mean_{window}s'] = (
            df[sensor].rolling(window=window).mean()
        )
        
        # Rolling std — stability over last N seconds
        df_features[f'{sensor}_std_{window}s'] = (
            df[sensor].rolling(window=window).std()
        )
        
        # Rate of change — how fast it moved
        df_features[f'{sensor}_roc'] = df[sensor].diff()
    
    df_features = df_features.dropna()
    return df_features

# Engineer features for strong sensors only
X_v3 = engineer_features_v2(df, strong_sensors, window=10)
y_true_v3 = y_true[X_v3.index]

print("Features used:", X_v3.shape[1])
print("Feature names:")
for col in X_v3.columns:
    print(f"  {col}")

Features used: 16
Feature names:
  Pressure
  Pressure_mean_10s
  Pressure_std_10s
  Pressure_roc
  Volume Flow RateRMS
  Volume Flow RateRMS_mean_10s
  Volume Flow RateRMS_std_10s
  Volume Flow RateRMS_roc
  Current
  Current_mean_10s
  Current_std_10s
  Current_roc
  Temperature
  Temperature_mean_10s
  Temperature_std_10s
  Temperature_roc


In [8]:
contamination_values = [0.20, 0.25, 0.30, 0.35, 0.40]

print("V3: Strong sensors only + engineered features")
print("="*55)
print(f"{'Contamination':<15} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("-"*55)

best_f1 = 0
best_contamination = 0
best_model = None

for cont in contamination_values:
    model_test = IsolationForest(contamination=cont, random_state=42)
    model_test.fit(X_v3)
    preds = model_test.predict(X_v3)
    y_pred_test = (preds == -1).astype(int)
    
    report = classification_report(y_true_v3, y_pred_test,
                                   output_dict=True)
    
    precision = report['1.0']['precision']
    recall = report['1.0']['recall']
    f1 = report['1.0']['f1-score']
    if f1 > best_f1:
        best_f1 = f1
        best_contamination = cont
        best_model = model_test
    
    print(f"{cont:<15} {precision:<12.3f} {recall:<10.3f} {f1:<10.3f}")

print(f"\nBest contamination: {best_contamination}")
print(f"Best F1: {best_f1:.3f}")
print(f"\nComparison:")
print(f"V1 (raw, all sensors):     F1=0.43")
print(f"V3 (engineered, 4 sensors): F1={best_f1:.3f}")

V3: Strong sensors only + engineered features
Contamination   Precision    Recall     F1        
-------------------------------------------------------
0.2             0.427        0.241      0.308     
0.25            0.405        0.286      0.335     
0.3             0.416        0.353      0.382     
0.35            0.412        0.408      0.410     
0.4             0.416        0.470      0.442     

Best contamination: 0.4
Best F1: 0.442

Comparison:
V1 (raw, all sensors):     F1=0.43
V3 (engineered, 4 sensors): F1=0.442


In [9]:
# Test window size 30 vs 10
print("Testing window sizes: 10s vs 30s")
print("="*55)
print(f"{'Window':<15} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("-"*55)

for window in [10, 30]:
    X_window = engineer_features_v2(df, strong_sensors, window=window)
    y_window = y_true[X_window.index]
    
    model_window = IsolationForest(contamination=0.40, random_state=42)
    model_window.fit(X_window)
    preds = model_window.predict(X_window)
    y_pred_window = (preds == -1).astype(int)
    
    report = classification_report(y_window, y_pred_window,
                                   output_dict=True)
    
    precision = report['1.0']['precision']
    recall = report['1.0']['recall']
    f1 = report['1.0']['f1-score']
    
    print(f"{window}s{'':<13} {precision:<12.3f} {recall:<10.3f} {f1:<10.3f}")

Testing window sizes: 10s vs 30s
Window          Precision    Recall     F1        
-------------------------------------------------------
10s              0.416        0.470      0.442     
30s              0.478        0.530      0.502     


## Key Finding — Day 3

Window size of 30s outperformed 10s (F1: 0.502 vs 0.442).

Initial hypothesis: shorter window better for brief faults.
Actual result: longer window better because this fault is sustained 
(lasts ~10 minutes not seconds).

Lesson: fault duration determines optimal window size. 
Always test rather than assume.

Best model so far:
- Strong sensors only: Pressure, Flow Rate, Current, Temperature
- Window: 30 seconds
- Contamination: 0.40
- F1: 0.502 (vs 0.43 baseline)